# 🧬 LifeOps: GRPO Training — Chaotic Life Management

Trains `Qwen2.5-3B-Instruct` with GRPO to balance Career, Family, and Health.

Run this notebook on **Colab T4 (free tier)**.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade "trl>=0.15.0" transformers datasets httpx matplotlib openenv-core python-dotenv mergekit

## 2. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print("✅ Model loaded")

## 3. Launch LifeOps Server (Robust Background)

In [ ]:
import os
import subprocess
import time
import httpx
import sys

# 1. Extract and Navigate
if os.path.exists("LifeOps.zip"):
    !unzip -o LifeOps.zip
    # Check if unzipped into a subfolder
    if os.path.exists("LifeOps") and os.path.isdir("LifeOps"):
        %cd LifeOps

# 2. Launch Server with Logging
print("🚀 Launching LifeOps Server...")
with open("server_log.txt", "w") as log:
    subprocess.Popen(["python", "-m", "server.app"], 
                     stdout=log, stderr=log, 
                     env=dict(os.environ, PYTHONPATH=os.getcwd()))

# 3. Retry loop for connection
ENV_URL = "http://localhost:8000"
connected = False
for i in range(10):
    time.sleep(3)
    try:
        with httpx.Client(base_url=ENV_URL, timeout=5) as http:
            health = http.get("/health")
            if health.status_code == 200:
                print(f"✅ LifeOps Server is LIVE at {ENV_URL}")
                connected = True
                break
    except:
        print(f"Attempt {i+1}: Waiting for server...")

if not connected:
    print("❌ Server failed. LOG OUTPUT:")
    if os.path.exists("server_log.txt"):
        with open("server_log.txt", "r") as f: print(f.read())

## 4. Define Reward Function

In [ ]:
import re

# Debug window for inspecting raw completions during training.
DEBUG_START_STEP = 3
DEBUG_END_STEP = 8
_reward_call_step = 0


def _extract_action_name(text: str) -> str | None:
    m = re.search(r"^\s*Action\s*:\s*([a-zA-Z0-9_ -]+)\s*$", text, re.IGNORECASE | re.MULTILINE)
    if not m:
        return None
    return re.sub(r"[\s-]+", "_", m.group(1).strip().lower())


def _coerce_prompt_text(prompt) -> str | None:
    if prompt is None:
        return None
    if isinstance(prompt, str):
        return prompt
    if isinstance(prompt, list) and prompt and isinstance(prompt[0], dict):
        chunks: list[str] = []
        for msg in prompt:
            role = str(msg.get("role", "")).strip()
            content = msg.get("content", "")
            if isinstance(content, str):
                chunks.append(f"{role}:\n{content}".strip())
            else:
                chunks.append(f"{role}:\n{str(content)}".strip())
        return "\n\n".join([c for c in chunks if c])
    return str(prompt)


def _parse_allowed_actions(prompt) -> set[str] | None:
    prompt_txt = _coerce_prompt_text(prompt)
    if not prompt_txt:
        return None
    m = re.search(r"Allowed Actions:\s*([^\n]+)", prompt_txt, re.IGNORECASE)
    if not m:
        return None
    parts = [p.strip() for p in m.group(1).split(",") if p.strip()]
    return set(parts) if parts else None


def _format_reward(text: str) -> float:
    """
    Dedicated format reward in roughly [-3, +2] range.
    This is intentionally stronger than the old tiny shaping term.
    """
    score = 0.0
    lines = [ln.rstrip() for ln in text.splitlines()]
    lines = [ln for ln in lines if ln.strip() != ""]

    # Must be exactly 2 non-empty lines
    if len(lines) == 2:
        score += 0.6
    else:
        score -= 1.2

    if len(lines) >= 1 and re.match(r"^\s*Action\s*:\s*\S", lines[0], re.IGNORECASE):
        score += 0.4
    else:
        score -= 0.8

    if len(lines) >= 2 and re.match(r"^\s*Justification\s*:\s*\S", lines[1], re.IGNORECASE):
        score += 0.4
    else:
        score -= 0.8

    if len(lines) >= 2:
        just_line = lines[1]
        m = re.match(r"^\s*Justification\s*:\s*(.+)\s*$", just_line, re.IGNORECASE)
        if m:
            body = m.group(1).strip()
            if "\n" in body:
                score -= 1.0
            wc = len(body.split())
            if wc > 28:
                score -= min(2.0, 0.08 * (wc - 28))
            if len(body) > 220:
                score -= min(2.0, 0.01 * (len(body) - 220))

    # Penalize common "essay" starters that usually mean rule-breaking verbosity
    if re.search(r"\b(however|therefore|moreover|additionally)\b", text, re.IGNORECASE):
        score -= 0.35

    return float(max(-3.0, min(2.0, score)))


def env_reward_fn(completions: list[str], prompts: list[str] | None = None, **kwargs) -> list[float]:
    global _reward_call_step
    _reward_call_step += 1

    # TRL may pass prompts positionally or via kwargs depending on version.
    if prompts is None:
        prompts = kwargs.get("prompts")

    rewards: list[float] = []
    with httpx.Client(base_url=ENV_URL, timeout=10) as http:
        for idx, completion in enumerate(completions):
            prompt = None
            if prompts is not None and idx < len(prompts):
                prompt = prompts[idx]

            allowed = _parse_allowed_actions(prompt)
            action_str = _extract_action_name(completion)

            if not action_str:
                rewards.append(-2.0)
                continue

            if allowed is not None and action_str not in allowed:
                rewards.append(-2.0)
                continue

            try:
                http.post("/reset")
                resp = http.post(
                    "/step",
                    json={"action": {"choice": action_str, "justification": "RL Step"}},
                )
                rewards.append(float(resp.json().get("reward", 0.0)) if resp.status_code == 200 else -2.0)
            except Exception:
                rewards.append(0.0)

    if DEBUG_START_STEP <= _reward_call_step <= DEBUG_END_STEP:
        print(f"\n===== DEBUG STEP {_reward_call_step} (env) =====")
        for i, completion in enumerate(completions[:5]):
            p = prompts[i] if prompts is not None and i < len(prompts) else None
            allowed = _parse_allowed_actions(p)
            action = _extract_action_name(completion)
            snippet = completion.replace("\n", "\\n")[:240]
            ok = (allowed is None) or (action in allowed) if action else False
            print(f"sample={i} action={action} allowed_ok={ok} env_r={rewards[i]:+.2f}")
            print(f"text: {snippet}")

    return rewards


def format_reward_fn(completions: list[str], **kwargs) -> list[float]:
    return [_format_reward(c) for c in completions]

## 5. Prepare Dataset

In [ ]:
from datasets import Dataset
import sys
import inspect
import copy

sys.path.append(os.getcwd())
from scripts.dataset_builder import LifeopsDatasetBuilder


def _truncate_messages(messages: list[dict], max_user_chars: int = 900) -> list[dict]:
    """Bound user message size for Colab stability."""
    msgs = copy.deepcopy(messages)
    for m in msgs:
        if m.get("role") == "user" and isinstance(m.get("content"), str):
            c = m["content"]
            if len(c) > max_user_chars:
                m["content"] = c[: max_user_chars - 30] + "\n\n[TRUNCATED]"
    return msgs


raw_items = LifeopsDatasetBuilder.generate_rl_dataset(100)
rows = []
for item in raw_items:
    rows.append({"prompt": _truncate_messages(item["prompt"])})

dataset = Dataset.from_list(rows)
print(f"✅ Generated {len(dataset)} examples")

try:
    from trl import GRPOConfig

    sig = inspect.signature(GRPOConfig.__init__)
    print("TRL GRPOConfig supports max_prompt_length:", "max_prompt_length" in sig.parameters)
    print("TRL GRPOConfig supports max_completion_length:", "max_completion_length" in sig.parameters)
    print("TRL GRPOConfig supports mask_truncated_completions:", "mask_truncated_completions" in sig.parameters)
    print("TRL GRPOConfig supports reward_weights:", "reward_weights" in sig.parameters)
except Exception as e:
    print("Could not introspect GRPOConfig:", repr(e))

## 6. GRPO Training

In [ ]:
import inspect
from trl import GRPOConfig, GRPOTrainer

# Newer Unsloth versions already patch GRPO internals automatically.
# On Colab / Python 3.12, explicit PatchFastRL() can raise OSError.
try:
    from unsloth import PatchFastRL
    try:
        PatchFastRL()
    except OSError:
        pass
except Exception:
    pass


def _build_grpo_config(**kwargs) -> GRPOConfig:
    """Build GRPOConfig across TRL versions (some kwargs vary by TRL release)."""
    sig = inspect.signature(GRPOConfig.__init__)
    filtered = {k: v for k, v in kwargs.items() if k in sig.parameters}
    dropped = sorted(set(kwargs) - set(filtered))
    if dropped:
        print("Note: dropping unsupported GRPOConfig args:", dropped)
    return GRPOConfig(**filtered)


cfg = _build_grpo_config(
    num_generations=4,
    max_steps=100,
    learning_rate=2e-5,
    max_prompt_length=384,  # only used on older TRL; ignored otherwise
    max_completion_length=72,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.08,
    mask_truncated_completions=True,
    reward_weights=[1.0, 0.45],
    output_dir="./grpo_out",
    logging_steps=1,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[env_reward_fn, format_reward_fn],
    args=cfg,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("GRPO config loaded:")
if hasattr(trainer.args, "max_prompt_length"):
    print("- max_prompt_length:", trainer.args.max_prompt_length)
else:
    print("- max_prompt_length: (not supported by this TRL version; prompts bounded in dataset cell)")
print("- max_completion_length:", getattr(trainer.args, "max_completion_length", None))
print("- num_generations:", trainer.args.num_generations)
print("- temperature:", getattr(trainer.args, "temperature", None))
print("- top_p:", getattr(trainer.args, "top_p", None))
print("- repetition_penalty:", getattr(trainer.args, "repetition_penalty", None))
print("- mask_truncated_completions:", getattr(trainer.args, "mask_truncated_completions", None))
print("- reward_weights:", getattr(trainer.args, "reward_weights", None))

gen_cfg = getattr(trainer.model, "generation_config", None)
if gen_cfg is not None:
    print("Model generation_config:")
    print("- max_new_tokens:", getattr(gen_cfg, "max_new_tokens", None))
    print("- eos_token_id:", getattr(gen_cfg, "eos_token_id", None))

trainer.train()